In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 30.4 MB/s eta 0:00:00


In [8]:
!pip install kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.7/34.7 MB 64.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.8 MB/s eta 0:00:00
  Created wheel for kiwipiepy_model: filename=kiwipiepy_model-0.20.0-py3-none-any.whl size=34818026 sha256=5de13019ba3168993340333162966894835bfd9310bcfe4945236785b941b1ea
  Stored in directory: /root/.cache/pip/wheels/ca/c8/52/3a539d6e9065b191fe1c215e0203dcc3e00601c0e3d3d39824
Successfully built kiwipiepy_model


In [ ]:
from kiwipiepy import Kiwi
from konlpy.tag import Hannanum
from konlpy.tag import Kkma
from konlpy.tag import Komoran
from konlpy.tag import Okt


kiwi = Kiwi()
hannanum = Hannanum()
kkma = Kkma()
komoran = Komoran()
okt = Okt()

In [ ]:
import pandas as pd

df =  pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna_trans.csv')

In [ ]:
from tqdm import tqdm
import re
stopwords = ['의','가','이','은','들','는','좀','잘','과','도','를','으로','자','에','와','한','하다',
             '적극', '소극','여부', '되다', '제', '매', '로', '때', '후', '로', '전', '민법', '방법',
             '경우', '상', '따르다', '있다', '않다', '원심', '및', '법', '에서', '또는', '그', '수', '에게',
             '인지', '해당', '에게', '위', '판결', '조', '인', '위', '사례', '사안', '대하', '되어다'
             '효력', '판단', '청구', '소송', '법원', '제기', '인정', '의미', '요건', '받다', '취지',
             '는지', '관하', '다고']


def tokenize_sentence(sentence):
    #result = kiwi.tokenize(sentence)
    tokenized_sentence = []
    #for r in result:
    #    tokenized_sentence.append(r[0])
    tokenized_sentence = komoran.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    l = ''
    for s in stopwords_removed_sentence:
        s = re.sub(r"[^가-힣\s]", " ", s)
        l += s +' '
    return l

In [ ]:
tokenized_q = [tokenize_sentence(q) for q in tqdm(df['question'])]
tokenized_trans_q = [tokenize_sentence(q) for q in tqdm(df['trans_question'])]

In [ ]:
df['trans_question'] = tokenized_trans_q
df['question'] = tokenized_q

In [ ]:
from gensim.models import Word2Vec

loaded_model = Word2Vec.load('/content/drive/MyDrive/파일위치/book_law_kkm_w2v.model')

In [ ]:
#임베딩하는 코드
##시용자 입력이 들어오면 (들어올 때마다) 사용자 입력과 전체 질문리스트(혹은 해당 카테고리의 질문 리스트)를 임베딩해줌
from tqdm import tqdm
def get_document_vectors(document_list):
    document_embedding_list = []
    #각 문서에 대해서
    for line in tqdm(document_list):
        doc2vec = None
        count = 0
        for word in line.split():
            #print(word)
            try:
                w = loaded_model.wv[word]
                count += 1

                #해당 문서에 있는 모든 단어들의 벡터 값을 더한다.
                if doc2vec is None :
                    doc2vec = w
                else :
                    doc2vec = doc2vec + w
            except:
                pass

        if doc2vec is not None:
            # 단어 벡터를 모두 더한 벡터의 값을 문서 길이로 나눠준다.
            doc2vec = doc2vec/count
            document_embedding_list.append(doc2vec)
        else:
            document_embedding_list.append(None)

    # 각 문서에 대한 문서 벡터 리스트를 리턴
    return document_embedding_list


In [ ]:
emd = get_document_vectors(df['question'])
emd2 = get_document_vectors(df['trans_question'])

In [ ]:
df['embedding']= emd
df['trans_q_embedding'] = emd2
df.to_csv('/content/drive/MyDrive/파일위치/pan_qna_trans_komoran_w2v.csv', index=False)